# A/B survey analysis

This notebook analyses responses from the blind A/B recipe survey (`remy.abtest`). In each survey, a participant:

1. rates a short cold-start quiz
2. sees two side-by-side lists of recommendations, one from each model, without knowing which model made which
3. rates every recipe in both lists, gives each list an overall score, and optionally says which list they prefer

Every survey contains **both** models for the **same person**, so every comparison here is **paired**. We test the within-person difference rather than comparing two independent groups.

**Unit of analysis.** A participant is identified by their invite-link id (`?pid=`) when one was given, and otherwise by the survey's own `user_id`. When someone retakes the survey, their runs are averaged so each person counts once.

**Data.** By default the notebook reads every file in `data/processed/abtest/responses/`. Each server start writes its own `responses_<timestamp>.jsonl` there. To analyse one deployment, or another folder such as simulated samples, set `REMY_ABTEST_RESPONSES` to a single file or a folder.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cycler import cycler
from matplotlib.ticker import MaxNLocator, PercentFormatter
from scipy import stats

from remy.abtest.responses import load_responses, read_surveys
from remy.paths import PROJECT_ROOT
from remy.plots import MODEL_COLORS, MODEL_LABELS

RESPONSES = Path(os.environ.get(
    "REMY_ABTEST_RESPONSES", PROJECT_ROOT / "data" / "processed" / "abtest" / "responses"
))

# Shared EDA palette and style (notebooks/eda.ipynb)
BACKGROUND, NEUTRAL, INK = "#EAEAEA", "#BDBBB6", "#333333"
MINT, PEACH, SAND, SKY = "#94D8A1", "#E1C6BA", "#E4D6AB", "#A9C8E8"
plt.rcParams.update({
    "figure.facecolor": BACKGROUND, "axes.facecolor": BACKGROUND, "savefig.facecolor": BACKGROUND,
    "axes.prop_cycle": cycler(color=[MINT, PEACH, SAND]),
    "figure.figsize": (9, 5), "figure.dpi": 120,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 14, "axes.labelpad": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#777777",
    "text.color": INK, "axes.labelcolor": INK, "xtick.color": "#555555", "ytick.color": "#555555",
    "xtick.major.size": 0, "ytick.major.size": 0,
    "lines.linewidth": 2.2, "legend.frameon": False, "legend.fontsize": 10,
})
BAR = dict(edgecolor="#555555", linewidth=0.6)
# Deeper strokes of the same hues, for lines where the pale fills would wash out
STROKE = {MINT: "#5DB873", PEACH: "#C4937E", SAND: "#C2AE6B", SKY: "#6E9CCB"}


def ygrid(ax):
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="white", linewidth=1.1)


def mean_ci(values, level=0.95):
    """Mean and t-based confidence half-width."""
    values = pd.Series(values).dropna()
    n = len(values)
    if n < 2:
        return values.mean(), np.nan
    return values.mean(), stats.t.ppf((1 + level) / 2, n - 1) * values.std(ddof=1) / np.sqrt(n)

## Load responses

- `surveys` has one row per completed survey. These are the raw JSON lines, from one file or from every file in the folder, with a `file` column naming the deployment each came from.
- `items` has one row per rated recommendation, built by `load_responses`.

If the survey config changed during the study (a different cutoff or model pair), only the most recent config is kept, because comparisons across configs aren't paired.

In [ ]:
surveys = read_surveys(RESPONSES)
print(surveys["file"].value_counts().sort_index().rename("surveys").to_string(), "\n")
surveys["config"] = surveys["train_cutoff"].astype(str) + " | " + surveys["models"].map(
    lambda m: " vs ".join(sorted(m))
)
print(surveys["config"].value_counts().to_string())

latest = surveys.sort_values("completed_at")["config"].iloc[-1]
surveys = surveys[surveys["config"] == latest].copy()
surveys["participant"] = surveys["external_id"].fillna(surveys["user_id"])
surveys["minutes"] = (
    pd.to_datetime(surveys["completed_at"]) - pd.to_datetime(surveys["started_at"])
).dt.total_seconds() / 60

items = load_responses(RESPONSES).merge(surveys[["user_id", "participant"]], on="user_id")

MODELS = sorted(items["model"].unique())
assert len(MODELS) == 2, MODELS
BASELINE = "popularity" if "popularity" in MODELS else MODELS[0]
CHALLENGER = next(m for m in MODELS if m != BASELINE)
COLORS = {BASELINE: MODEL_COLORS.get(BASELINE, MINT), CHALLENGER: MODEL_COLORS.get(CHALLENGER, PEACH)}
LABEL = {m: MODEL_LABELS.get(m, m) for m in MODELS}
print(f"\nAnalysing: {latest}   (challenger = {LABEL[CHALLENGER]}, baseline = {LABEL[BASELINE]})")

## Overview

In [ ]:
n_participants = surveys["participant"].nunique()
pd.Series({
    "Completed surveys": len(surveys),
    "Participants": n_participants,
    "Participants who retook it": int((surveys["participant"].value_counts() > 1).sum()),
    "Quiz recipes per survey": surveys["survey"].iloc[0]["n_quiz"],
    "Recipes per list": surveys["survey"].iloc[0]["n_recs"],
    "Median minutes to complete": round(surveys["minutes"].median(), 1),
    "First response": str(surveys["completed_at"].min())[:16],
    "Last response": str(surveys["completed_at"].max())[:16],
}, name="").to_frame()

## Paired comparison

Each participant gets three scores per model:

- `mean_item`: the mean star rating of the recipes in that model's list
- `list_rating`: the overall 1–5 score they gave the list
- `share_4plus`: the share of that list's recipes rated 4 or 5 stars, i.e. the hit rate of recipes they'd actually want to make

The difference is challenger minus baseline. The table reports:

- a 95% confidence interval for that difference
- a paired *t*-test
- a Wilcoxon signed-rank test, which is the safer test for ordinal star ratings
- Cohen's *d<sub>z</sub>*: the mean difference divided by the standard deviation of the differences

In [ ]:
METRICS = {
    "mean_item": "Mean recipe rating",
    "list_rating": "Overall list rating",
    "share_4plus": "Share of recipes rated 4+",
}
per_survey = (
    items.groupby(["participant", "user_id", "model", "side"])
    .agg(mean_item=("rating", "mean"), list_rating=("list_rating", "first"),
         share_4plus=("rating", lambda r: (r >= 4).mean()))
    .reset_index()
)
# Average retakes so that each participant counts once
paired = per_survey.groupby(["participant", "model"])[list(METRICS)].mean().unstack("model")
diffs = pd.DataFrame({m: paired[(m, CHALLENGER)] - paired[(m, BASELINE)] for m in METRICS})


def paired_test(metric):
    a, b = paired[(metric, CHALLENGER)], paired[(metric, BASELINE)]
    d = a - b
    mean, half = mean_ci(d)
    try:
        p_wilcoxon = stats.wilcoxon(a, b).pvalue
    except ValueError:  # all differences are zero
        p_wilcoxon = np.nan
    return {
        "metric": METRICS[metric], "n": len(d),
        LABEL[BASELINE]: b.mean(), LABEL[CHALLENGER]: a.mean(),
        "difference": mean, "95% CI low": mean - half, "95% CI high": mean + half,
        "Cohen's dz": mean / d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan,
        "paired t p": stats.ttest_rel(a, b).pvalue, "Wilcoxon p": p_wilcoxon,
    }


results = pd.DataFrame([paired_test(m) for m in METRICS]).set_index("metric")
results.style.format(precision=3).format({"n": "{:d}"})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6))
for ax, metric in zip(axes, METRICS):
    for i, model in enumerate((BASELINE, CHALLENGER)):
        mean, half = mean_ci(paired[(metric, model)])
        ax.bar(i, mean, width=0.6, color=COLORS[model], **BAR)
        ax.errorbar(i, mean, yerr=half, color=INK, capsize=5, linewidth=1.4)
        label = f"{mean:.0%}" if metric == "share_4plus" else f"{mean:.2f}"
        top = mean + (half if np.isfinite(half) else 0)
        ax.annotate(label, (i, top), ha="center", va="bottom", fontsize=10,
                    xytext=(0, 4), textcoords="offset points")
    ax.set_xticks([0, 1], [LABEL[BASELINE], LABEL[CHALLENGER]])
    ax.set_title(METRICS[metric], fontsize=12)
    if metric == "share_4plus":
        ax.set_ylim(0, 1.08)
        ax.yaxis.set_major_formatter(PercentFormatter(1))
    else:
        ax.set_ylim(1, 5.3)
    ygrid(ax)
fig.suptitle("Average score per participant, with 95% confidence intervals",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()

### Distribution of within-person differences

Each bar counts participants. Bars to the right of zero are people who rated the challenger's list higher. A tight cluster on one side means the effect is consistent; a wide spread means the average is driven by a few people.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, metric, width in zip(axes, ["mean_item", "list_rating"], [0.25, 0.5]):
    d = diffs[metric].dropna()
    lim = max(1.0, np.ceil(d.abs().max() / width) * width)
    bins = np.arange(-lim - width / 2, lim + width, width)
    counts, edges = np.histogram(d, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    colors = [COLORS[CHALLENGER] if c > 1e-9 else COLORS[BASELINE] if c < -1e-9 else NEUTRAL for c in centers]
    ax.bar(centers, counts, width=width * 0.92, color=colors, **BAR)
    mean, half = mean_ci(d)
    ax.axvspan(mean - half, mean + half, color=INK, alpha=0.08, linewidth=0)
    ax.axvline(0, color="#777777", linewidth=1)
    ax.axvline(mean, color=INK, linewidth=1.6, linestyle="--")
    ax.annotate(f"mean {mean:+.2f}", (mean, counts.max()), xytext=(6, 0),
                textcoords="offset points", fontsize=10, va="top")
    ax.text(0.98, 0.97, f"favours {LABEL[CHALLENGER]} →", transform=ax.transAxes, ha="right", va="top", fontsize=10)
    ax.text(0.02, 0.97, f"← favours {LABEL[BASELINE]}", transform=ax.transAxes, ha="left", va="top", fontsize=10)
    ax.set_xlabel(f"{METRICS[metric]}: {LABEL[CHALLENGER]} − {LABEL[BASELINE]}")
    ax.set_ylabel("Participants")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylim(0, counts.max() * 1.25)
    ygrid(ax)
fig.suptitle("Within-person difference between the two lists", x=0.01, ha="left",
             fontsize=13, fontweight="bold")
fig.tight_layout()

## Head-to-head preference

This asks: if participants could only cook from one list, which one would they pick? The question is optional (`survey.ask_preference` in the config); this section is skipped when it was off. A two-sided sign (binomial) test compares the wins, ignoring "no preference". Unlike the averages above, this counts each **survey**, so retakes count separately. Only one test here counts surveys rather than participants, so keep that in mind if many people retook it.

In [ ]:
asked = surveys["survey"].map(lambda c: c.get("ask_preference", True))
if not asked.any():
    print("The preference question was turned off (survey.ask_preference: false), so there is nothing to show.")
else:
    surveys_asked = surveys[asked]
    pref = surveys_asked["preferred_model"].fillna("none").value_counts()
    wins_c, wins_b, ties = pref.get(CHALLENGER, 0), pref.get(BASELINE, 0), pref.get("none", 0)
    total = wins_c + wins_b + ties
    sign_p = stats.binomtest(wins_c, wins_c + wins_b).pvalue if wins_c + wins_b else np.nan

    fig, ax = plt.subplots(figsize=(9, 2.4))
    left = 0
    for label, count, color in [(LABEL[CHALLENGER], wins_c, COLORS[CHALLENGER]),
                                ("No preference", ties, NEUTRAL),
                                (LABEL[BASELINE], wins_b, COLORS[BASELINE])]:
        share = count / total
        ax.barh(0, share, left=left, height=0.55, color=color, **BAR)
        if share > 0.06:
            ax.text(left + share / 2, 0, f"{label}\n{share:.0%} ({count})", ha="center", va="center", fontsize=10)
        left += share
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.spines["left"].set_visible(False)
    ax.set_title(f"Which list would you rather cook from?   (n = {total} surveys, sign test p = {sign_p:.3f})")
    fig.tight_layout()

## Star rating distribution

In [ ]:
stars = np.arange(1, 6)
share = (items.groupby("model")["rating"].value_counts(normalize=True)
         .unstack(fill_value=0).reindex(columns=stars, fill_value=0))
quiz = pd.DataFrame([q for qs in surveys["quiz"] for q in qs])
quiz_share = quiz["rating"].value_counts(normalize=True).reindex(stars, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4.8))
w = 0.27
for offset, (name, values, color) in zip(
    (-w, 0, w),
    [("Quiz (cold start)", quiz_share, NEUTRAL),
     (LABEL[BASELINE], share.loc[BASELINE], COLORS[BASELINE]),
     (LABEL[CHALLENGER], share.loc[CHALLENGER], COLORS[CHALLENGER])],
):
    ax.bar(stars + offset, values, width=w * 0.92, color=color, label=name, **BAR)
ax.set_xticks(stars, [f"{s} ★" for s in stars])
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_ylabel("Share of ratings")
ax.set_title("How recipes were rated")
ax.legend(loc="upper left")
ygrid(ax)
fig.tight_layout()

## Does each model put its best recipes first?

These are mean ratings by rank position. A model that ranks well should show a downward slope: its top-ranked recipes get the highest ratings. The shaded bands are 95% confidence intervals across surveys.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for model in (BASELINE, CHALLENGER):
    by_rank = items[items["model"] == model].groupby("rank")["rating"]
    summary = by_rank.apply(lambda r: pd.Series(mean_ci(r), index=["mean", "half"])).unstack()
    ax.plot(summary.index, summary["mean"], color=STROKE.get(COLORS[model], COLORS[model]),
            marker="o", markersize=7, markerfacecolor=COLORS[model],
            markeredgecolor="#555555", markeredgewidth=0.6, label=LABEL[model])
    ax.fill_between(summary.index, summary["mean"] - summary["half"], summary["mean"] + summary["half"],
                    color=COLORS[model], alpha=0.25, linewidth=0)
    ax.annotate(LABEL[model], (summary.index[-1], summary["mean"].iloc[-1]), xytext=(8, 0),
                textcoords="offset points", va="center", fontsize=10)
ax.set_xlabel("Rank in list")
ax.set_ylabel("Mean star rating")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title("Mean rating by position in the list")
ax.legend(loc="lower left")
ygrid(ax)
fig.tight_layout()

## Sanity checks

**Position bias.** Which model appears as List 1 (left) is random, so position shouldn't change the model comparison. Still, if participants systematically favour the left list, that adds noise. This compares left and right for each participant, regardless of which model was on each side.

In [ ]:
side_paired = per_survey.groupby(["participant", "side"])[["mean_item", "list_rating"]].mean().unstack("side")
pd.DataFrame([
    {"metric": METRICS[m], "List 1 (left)": side_paired[(m, "left")].mean(),
     "List 2 (right)": side_paired[(m, "right")].mean(),
     "paired t p": stats.ttest_rel(side_paired[(m, "left")], side_paired[(m, "right")]).pvalue}
    for m in ["mean_item", "list_rating"]
]).set_index("metric").style.format(precision=3)

**Do the overall list scores track the item ratings?** If they do, the overall score is a reliable summary. Big disagreements suggest participants judged the lists on something else, such as variety. Points are jittered slightly so overlapping ones stay visible.

In [ ]:
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(7, 6))
for model in (BASELINE, CHALLENGER):
    s = per_survey[per_survey["model"] == model]
    ax.scatter(s["mean_item"], s["list_rating"] + rng.uniform(-0.12, 0.12, len(s)),
               s=46, color=COLORS[model], edgecolor="#555555", linewidth=0.6, alpha=0.9, label=LABEL[model])
r = stats.spearmanr(per_survey["mean_item"], per_survey["list_rating"]).statistic
ax.plot([1, 5], [1, 5], color="#777777", linewidth=1, linestyle=":")
ax.set_xlim(0.8, 5.2)
ax.set_ylim(0.6, 5.4)
ax.set_xlabel("Mean recipe rating in the list")
ax.set_ylabel("Overall list rating")
ax.set_title(f"Overall list score vs. its recipes   (Spearman ρ = {r:.2f})")
ax.legend(loc="upper left")
ygrid(ax)
fig.tight_layout()

**How different are the two lists?** When both models recommend mostly the same recipes, the participant can't tell the lists apart, and the differences above shrink toward zero.

In [ ]:
overlap = items.groupby(["user_id", "model"])["recipe_id"].agg(set).unstack("model")
overlap = overlap.apply(lambda r: len(r[BASELINE] & r[CHALLENGER]), axis=1)
n_recs = surveys["survey"].iloc[0]["n_recs"]
counts = overlap.value_counts().reindex(range(n_recs + 1), fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(counts.index, counts.values, width=0.8, color=SKY, **BAR)
ax.set_xticks(range(n_recs + 1))
ax.set_xlabel(f"Recipes appearing in both lists (out of {n_recs})")
ax.set_ylabel("Surveys")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title(f"Overlap between the two lists   (median {overlap.median():.0f} shared)")
ygrid(ax)
fig.tight_layout()

**Is each model personalising?** This counts how many distinct ordered lists each model produced across surveys. A model that gives every participant the same list isn't using their quiz answers. Popularity is expected to do this; a personalised model shouldn't.

In [ ]:
lists = (items.sort_values("rank").groupby(["model", "user_id"])["recipe_id"].agg(tuple)
         .groupby("model").agg(surveys="size", distinct_lists="nunique"))
lists["distinct recipes shown"] = items.groupby("model")["recipe_id"].nunique()
lists.index = lists.index.map(LABEL)
lists

## Most-shown recipes per model

In [ ]:
top = (items.groupby(["model", "recipe_id", "name"])["rating"]
       .agg(times_shown="size", mean_rating="mean").reset_index()
       .sort_values(["model", "times_shown", "mean_rating"], ascending=[True, False, False])
       .groupby("model").head(8))
top["model"] = top["model"].map(LABEL)
top.set_index(["model", "name"])[["times_shown", "mean_rating"]].style.format({"mean_rating": "{:.2f}"})

## Reading the results

- **Headline.** Use the paired difference in mean recipe rating and in the share of recipes rated 4+, together with the preference sign test when that question was asked. Report the mean difference with its 95% confidence interval, not just the *p*-value.
- **Ordinal ratings.** Star ratings are ordinal, so the Wilcoxon test is the safer test. If the *t*-test and Wilcoxon disagree, trust Wilcoxon and look at the distribution of differences.
- **Several tests at once.** Three metrics plus the preference test means several tests. Decide on a primary metric before looking at the data, or apply a Holm correction.
- **Retakes.** The paired tests average retakes per participant. The preference test counts surveys.
- **Overlap and personalisation.** The comparison only says something when the lists actually differ. If the overlap is high, or a model shows only a handful of distinct lists, the test is mostly comparing two fixed rankings rather than personalisation.